In [ ]:
import numpy as np
import pandas as pd
from py_vollib.black_scholes_merton.implied_volatility import implied_volatility#, BelowIntrinsicException
from py_lets_be_rational.exceptions import BelowIntrinsicException

# Define the symbol and option types
symbol = 'AAPL'
option_types = ['call', 'put']

# Define the strike prices to generate options for
strike_prices = np.arange(100, 200, 10)

# Define the expiration dates to generate options for
expiration_dates = pd.date_range(start='2020-06-01', end='2022-06-01', freq='D')

# Initialize an empty DataFrame to store the simulated option data
option_quotes = pd.DataFrame(columns=['Symbol', 'Expiration', 'Type', 'Strike', 'Last', 'Change', 'Bid', 'Ask', 'Volume', 'Open Interest', 'Implied Volatility'])

# Loop through each expiration date and strike price to generate option quotes
for expiration in expiration_dates:
    for option_type in option_types:
        for strike_price in strike_prices:
            # Generate random values for each field except for last
            bid = np.random.uniform(strike_price - 10, strike_price - 1)
            ask = np.random.uniform(strike_price + 1, strike_price + 10)
            volume = np.random.randint(0, 1000)
            open_interest = np.random.randint(0, 5000)
            # Calculate the implied volatility using the Black-Scholes model
            underlying_price = np.random.uniform(strike_price - 10, strike_price + 10)
            risk_free_rate = 0.02
            time_to_expiry = (expiration - pd.Timestamp.today()).days / 365
            intrinsic_value = max(0, underlying_price - strike_price) if option_type == 'call' else max(0, strike_price - underlying_price)
            for i in range(10):
                last = intrinsic_value + np.random.uniform(0.01, 0.5) * (ask - intrinsic_value)
                if last > intrinsic_value:
                    try:
                        implied_vol = implied_volatility(price=last, S=underlying_price, strike=strike_price,
                                                         t=time_to_expiry, r=risk_free_rate, flag=option_type[0], q=0)
                    except:# BelowIntrinsicException:
                        implied_vol = np.nan
                    # Add the option quote to the DataFrame
                    option_quotes = option_quotes.append({
                        'Symbol': symbol,
                        'Expiration': expiration,
                        'Type': option_type,
                        'Strike': strike_price,
                        'Last': last,
                        'Change': ask - last,
                        'Bid': bid,
                        'Ask': ask,
                        'Volume': volume,
                        'Open Interest': open_interest,
                        'Implied Volatility': implied_vol
                    }, ignore_index=True)

# Print the first 10 rows of the option quotes DataFrame
print(option_quotes.head(10))


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import datetime

def options_chain(symbol):

    tk = yf.Ticker(symbol)
    # Expiration dates
    exps = tk.options
    print(exps)

    # Get options for each expiration
    options = pd.DataFrame()
    for e in exps:
        opt = tk.option_chain(e)
        opt = pd.DataFrame().append(opt.calls).append(opt.puts)
        opt['expirationDate'] = e
        options = options.append(opt, ignore_index=True)

    # Bizarre error in yfinance that gives the wrong expiration date
    # Add 1 day to get the correct expiration date
    options['expirationDate'] = pd.to_datetime(options['expirationDate']) + datetime.timedelta(days = 1)
    options['dte'] = (options['expirationDate'] - datetime.datetime.today()).dt.days / 365
    
    # Boolean column if the option is a CALL
    options['CALL'] = options['contractSymbol'].str[4:].apply(
        lambda x: "C" in x)
    
    options[['bid', 'ask', 'strike']] = options[['bid', 'ask', 'strike']].apply(pd.to_numeric)
    options['mark'] = (options['bid'] + options['ask']) / 2 # Calculate the midpoint of the bid-ask
    
    # Drop unnecessary and meaningless columns
    options = options.drop(columns = ['contractSize', 'currency', 'change', 'percentChange', 'lastTradeDate', 'lastPrice'])

    return options

In [ ]:
df = options_chain("^spx")

In [ ]:
df

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import scipy.stats as si

In [ ]:
df.strike.values

In [ ]:
df["impliedVolatility"][1]

In [ ]:
S = df.strike.values.astype(np.float64)
T = df.dte.values.astype(np.float64)
ImpliedVolatility = np.zeros((len(T),len(S)))

for j in range(len(S)):
    for i in range(len(T)):
        ImpliedVolatility[i,j] = df["impliedVolatility"][j]

In [ ]:
ImpliedVolatility=ImpliedVolatility.astype(np. float64)

In [ ]:
ImpliedVolatility.plot()

In [ ]:
fig = plt.figure(figsize= (15, 8))
ax = fig.gca(projection='3d')
S, T = np.meshgrid(S, T)
surf = ax.plot_surface(S, T, ImpliedVolatility, rstride=2, cstride=2, cmap=plt.cm.coolwarm, linewidth=0.5, antialiased=True)
ax.set_xlabel('Stock Price')
ax.set_ylabel('Time to Expiry')
ax.set_zlabel('ImpliedVolatility')
fig.colorbar(surf, shrink=0.5, aspect=5);

In [ ]:
st = S
ttm=T
a2=ImpliedVolatility
fig = plt.figure(figsize= (15, 15))
ax = fig.gca(projection='3d')
surf = ax.plot_surface(st, ttm, a2, rstride=2, cstride=2, cmap=plt.cm.coolwarm, linewidth=0.1, antialiased=True)
ax.set_xlabel('Strike')
ax.set_ylabel('Time to Maturitre')
ax.set_zlabel('Volatile Implied')
fig.colorbar(surf, shrink=0.5, aspect=1,);